# 02 · A positive-unlabeled classifier

Notebook 01 ranked candidates by similarity to my history, with nothing trained. This one trains something and asks whether it beats that, on the same split and the same candidates.

Everything on what.pm is something I liked, so there are only positives. The way around that is positive-unlabeled learning: a random sample of the catalog is treated as negative. Most of it really is things I'd never pick, some of it is things I'd love, and the model has to put up with that noise.

The code is in `src/recs/classify.py`; `uv run classify --rolling` prints the table at the end without the walkthrough.

In [ ]:
import numpy as np
import pandas as pd

from recs.classify import Features, fit_logistic, fit_mlp, ranks_of_heldout, evaluate
from recs.config import default_model_key
from recs.evaluate import ROLLING, HOLDOUT, summarise, temporal_split, is_complete
from recs.paths import ITEMS, catalog_path

pd.set_option("display.width", 140)
items = pd.read_parquet(ITEMS)
model_key = default_model_key()
split = temporal_split(items)
print(model_key, "| history", split.train.sum(), "| held out", split.heldout.sum(), "| cut", split.cut_date)

## Features

Per item: its content vector (1024 numbers from the embedding model), plus one more: its recency-weighted similarity to my history, the baseline's own score. A history item is nearest to itself, so for the positives that score is taken against the rest of the history.

Two things are deliberately left out, because the first version of this learned them instead of my taste:

- Items without a synopsis. A fifth of mine have a bare title-and-author text and the catalog almost never does, so the classifier learned "short text = hers" and put every such held-out book at the top. Training uses fully described items only, positives and negatives alike.
- MovieLens. Most of my films are in it and the recent ones aren't, and a "known to MovieLens" flag is the same kind of shortcut. It's blended in after scoring, as in notebook 01.

In [ ]:
rng = np.random.default_rng(0)
features = Features(items, split, model_key, "book", k=5, half_life=3.0)
x, y, weight = features.training_set(negatives_per_positive=5, rng=rng)
print("training rows:", x.shape, "| positives:", int(y.sum()), "| features:", x.shape[1], "(content vector + similarity)")
print("candidates:", features.candidates.shape, "of which held out:", len(features.heldout))

## Logistic regression

Linear, fast, and its weights say which directions of the embedding space mean "mine". If this doesn't beat the baseline the setup is wrong, not the model. The positives are weighted by recency like the baseline, and `class_weight="balanced"` stops the five-to-one negatives from drowning them.

In [ ]:
predict = fit_logistic(x, y, weight)
ranks = ranks_of_heldout(features.blended(predict(features.candidates)), len(features.heldout))
baseline_ranks = ranks_of_heldout(features.blended(features.candidates[:, -1]), len(features.heldout))
n_candidates = len(features.candidates)
pd.DataFrame({"similarity": summarise(baseline_ranks, n_candidates), "logistic": summarise(ranks, n_candidates)}).round(3)

## A small MLP

Two hidden layers, dropout, two hundred epochs of full-batch Adam on a few thousand rows. It can draw a boundary that isn't a straight line through the embedding space; the question is whether that helps or just fits the noise in the presumed negatives.

In [ ]:
predict_mlp = fit_mlp(x, y, weight)
ranks_mlp = ranks_of_heldout(features.blended(predict_mlp(features.candidates)), len(features.heldout))
pd.DataFrame({"similarity": summarise(baseline_ranks, n_candidates), "logistic": summarise(ranks, n_candidates), "mlp": summarise(ranks_mlp, n_candidates)}).round(3)

## All media, five cut dates

The same, per medium, averaged over the rolling windows from notebook 01, for held-out items with a full text. Films are blended with MovieLens in every row.

In [ ]:
table = evaluate(model_key, ROLLING, negatives_per_positive=5, k=5, half_life=3.0, complete_only=True, methods=("similarity", "logistic", "mlp"))
table.set_index(["medium", "method"]).round(3)

## Caveats

The year is in the text, and the classifier can read it. If the catalog is older than what I read, "recent" alone is a strong signal, which is why the book catalog has a recent-years pass. `evaluate` and this notebook can't tell taste from recency apart on their own; the check is whether a newest-first ordering gets close to the same numbers.

The presumed negatives are a sample, so the numbers move a little between runs. The seed is fixed here, and a sweep over `--negatives` is the obvious next experiment.

Nothing here is tuned on the hold-out except the half-life, inherited from the baseline. The regularisation and the MLP's size are first guesses.